In [6]:
import pandas as pd
import xarray as xr
import numpy as np
#import dask.array as da
#import dask.dataframe as dd
import cftime
import os,warnings
warnings.filterwarnings('ignore') # Turns off warnings

def fast_csv_to_netcdf(input_csv, output_nc):
    # Read CSV using dask
    df = pd.read_csv(input_csv, 
                     dtype={
                         'X': 'float32', 
                         'Y': 'float32', 
                         'Value': 'float32', 
                         'Month': 'int32', 
                         'Year': 'int32'
                     })
    print(df)
    # Get unique coordinates
    #x_coords = sorted(df['X'].unique().compute())
    #y_coords = sorted(df['Y'].unique().compute())
    
    # Get time coordinates more efficiently
    #time_df = df[['Year', 'Month']].drop_duplicates().compute()
    #time_coords = sorted([cftime.Datetime360Day(y, m, 1) 
    #                     for y, m in zip(time_df['Year'], time_df['Month'])])
    
    # Compute the DataFrame
    #df_computed = df.compute()
    
    # Handle duplicates by averaging values for the same coordinates
    #df_grouped = df_computed.groupby(['Year', 'Month', 'Y', 'X'])['Value'].mean()
    df["time"] = pd.to_datetime(
              dict(year=df['Year'], month=df["Month"], day=1)
    )
    df = df.set_index(["time", "Y", "X"])
    ds=df.to_xarray()
    print(ds)
    
    # Create a 3D numpy array filled with NaN
    #shape = (len(time_coords), len(y_coords), len(x_coords))
    #data_array = np.full(shape, np.nan, dtype='float32')
    
    # Create lookup dictionaries for faster indexing
    #time_lookup = {(d.year, d.month): i for i, d in enumerate(time_coords)}
    #y_lookup = {y: i for i, y in enumerate(y_coords)}
    #x_lookup = {x: i for i, x in enumerate(x_coords)}
    
    # Fill the array with values
    #for (year, month, y, x), value in df_grouped.items():
    #    t_idx = time_lookup.get((year, month))
    #    y_idx = y_lookup.get(y)
    #    x_idx = x_lookup.get(x)
    #    if all(idx is not None for idx in (t_idx, y_idx, x_idx)):
    #        data_array[t_idx, y_idx, x_idx] = value
    
    # Create xarray Dataset directly
    #ds = xr.Dataset(
    #    data_vars={
    #        'snow_water_equivalent': (['time', 'Y', 'X'], data_array)
    #    },
    #    coords={
    #        'time': time_coords,
    #        'Y': y_coords,
    #        'X': x_coords
    #    }
    #)
    
    # Add metadata
   # ds.snow_water_equivalent.attrs = {
   #     'units': 'mm',
   #     'long_name': 'Snow Water Equivalent'
   # }
    
    # Save with efficient encoding
#    encoding = {
#        'time': {
#            'calendar': '360_day',
#            'calendar': 'gregorian',
#            'units': 'days since 1900-01-01'
#        },
#        'snow_water_equivalent': {
#            'zlib': True,
#            'complevel': 5,
#            'chunksizes': (1, len(y_coords), len(x_coords))
#        }
#    }
    
#    ds.to_netcdf(output_nc, encoding=encoding)
    return ds

# Usage
if __name__ == '__main__':
    input_csv = '/data/esplab/aelyoussoufi/combined_snow_water.csv'
    output_nc = '../data/swe/processed/snow_water_equivalent.nc'
    ds = fast_csv_to_netcdf(input_csv, output_nc)
    print(ds)

                     X          Y      Value  Year  Month
0           -99.679581  50.120419        NaN  2003     12
1          -107.571251  25.437084        NaN  2003     12
2          -111.371246  49.062084   0.000000  2003     12
3           -83.887917  52.503750        NaN  2003     12
4           -71.929581  27.520416        NaN  2003     12
...                ...        ...        ...   ...    ...
1501414346  -75.796249  25.928749        NaN  2020     12
1501414347  -85.687920  49.045418        NaN  2020     12
1501414348 -105.146248  30.312084        NaN  2020     12
1501414349  -72.329582  42.995419  10.870968  2020     12
1501414350  -79.929581  38.537083  21.741936  2020     12

[1501414351 rows x 5 columns]


ValueError: cannot convert a DataFrame with a non-unique MultiIndex into xarray

In [5]:
df

NameError: name 'df' is not defined